In [ ]:
from huggingface_hub import hf_hub_download, upload_file, create_repo

SOURCE_REPO = "srijan2715/brisc2025-paper-pipeline"
TARGET_REPO = "srijan2715/scratchcnn-checkpoints" 

# 1. Automatically create the target Model repo if it doesn't exist
print(f"⚙️ Ensuring target repo '{TARGET_REPO}' exists...")
create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True)
print("✅ Target repo is ready!\n")

files_to_move = [
    ("scratch_paper/scratch_paper_brisc_seed42.pth", "brisc_seed42.pth"),
    ("scratch_paper/scratch_paper_brisc_seed43.pth", "brisc_seed43.pth"),
    ("scratch_paper/scratch_paper_brisc_seed44.pth", "brisc_seed44.pth"),
    ("scratch_paper/scratch_paper_sartaj_seed42.pth", "sartaj_seed42.pth"),
    ("scratch_paper/scratch_paper_sartaj_seed43.pth", "sartaj_seed43.pth"),
    ("scratch_paper/scratch_paper_sartaj_seed44.pth", "sartaj_seed44.pth"),
]

print("🚀 Starting secure file transfer...\n")

for old_name, new_name in files_to_move:
    print(f"⏳ Moving {old_name} -> {new_name}")
    try:
        # Download from Dataset repo
        local_path = hf_hub_download(
            repo_id=SOURCE_REPO, 
            filename=old_name, 
            repo_type="dataset" 
        )
        
        # Upload to Model repo
        upload_file(
            path_or_fileobj=local_path,
            path_in_repo=new_name,
            repo_id=TARGET_REPO,
            repo_type="model"
        )
        print(f"✅ Successfully uploaded {new_name}")
    except Exception as e:
        print(f"❌ Failed to move {old_name}. Error: {e}")

print("\n🎉 ALL 6 MODELS SUCCESSFULLY MOVED! 🎉")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "srijan2715/brisc2025-paper-pipeline"

files_to_delete = [
    "scratch_paper/scratch_paper_brisc_seed42.pth",
    "scratch_paper/scratch_paper_brisc_seed43.pth",
    "scratch_paper/scratch_paper_brisc_seed44.pth",
    "scratch_paper/scratch_paper_sartaj_seed42.pth",
    "scratch_paper/scratch_paper_sartaj_seed43.pth",
    "scratch_paper/scratch_paper_sartaj_seed44.pth",
]

print("🗑️ Deleting redundant files from Dataset repo...")
for file in files_to_delete:
    try:
        api.delete_file(path_in_repo=file, repo_id=repo_id, repo_type="dataset")
        print(f"✅ Deleted {file}")
    except Exception as e:
        print(f"⚠️ Could not delete {file} (might already be gone): {e}")

print("\n🎉 Dataset repo is now clean!")

In [ ]:
import torch
from huggingface_hub import hf_hub_download

REPO_ID = "srijan2715/scratchcnn-checkpoints"

files_to_check = [
    "brisc_seed42.pth", "brisc_seed43.pth", "brisc_seed44.pth",
    "sartaj_seed42.pth", "sartaj_seed43.pth", "sartaj_seed44.pth"
]

print("🔍 Inspecting checkpoints for saved training metrics...\n")

for filename in files_to_check:
    print(f"--- Checking {filename} ---")
    try:
        # Download the file
        local_path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type="model")
        
        # Load the checkpoint
        checkpoint = torch.load(local_path, map_location="cpu")
        keys = list(checkpoint.keys())
        
        print(f"📦 Keys found: {keys[:5]}...")
        
        # Look for common metric names
        metrics_found = False
        for key in ['accuracy', 'acc', 'val_accuracy', 'val_acc', 'metrics', 'epoch', 'train_acc']:
            if key in checkpoint:
                print(f"  ✅ FOUND METRIC: {key} = {checkpoint[key]}")
                metrics_found = True
                
        if not metrics_found:
            if any("weight" in str(k) or "bias" in str(k) for k in keys):
                print("  ℹ️ File contains ONLY model weights. No training metrics saved.")
            else:
                print(f"  ℹ️ Unrecognized format. Full keys: {keys}")
                
    except Exception as e:
        print(f"  ❌ Error loading {filename}: {e}")
    print("")

print("🏁 Inspection complete!")

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json

# 1. Check the Dataset's Native Features
print("--- DATASET NATIVE LABELS ---")
dataset = load_dataset("srijan2715/brisc2025-paper-pipeline", split="train")
print("Dataset Features:", dataset.features)
print("First 5 raw labels:", dataset['label'][:5])
print("First 5 raw images (shape):", [type(img) for img in dataset['image'][:5]])

# 2. Read the Label Mapping JSON
print("\n--- LABEL MAPPING JSON ---")
try:
    path = hf_hub_download(repo_id="srijan2715/brisc2025-paper-pipeline", filename="label_mapping1.json", repo_type="dataset")
    with open(path, 'r') as f:
        mapping = json.load(f)
    print("Exact Mapping Used in Training:", mapping)
except Exception as e:
    print("Could not find label_mapping1.json:", e)

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import gc

# ==========================================
# STEP 1: Architecture (Verified & Fixed)
# ==========================================
class ScratchCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ScratchCNN, self).__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# ==========================================
# STEP 2: Configuration (FIXED LABEL MAPPING!)
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {DEVICE}\n")

# Bulletproof mapping to handle both 'no' and 'no_tumor'
CLASS_TO_IDX = {'glioma': 0, 'meningioma': 1, 'no': 2, 'no_tumor': 2, 'pituitary': 3}

# Display names for the final report (mapped to indices 0, 1, 2, 3)
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]

REPO_ID = "srijan2715/scratchcnn-checkpoints"
DATASETS = {
    "BRISC": "srijan2715/brisc2025-paper-pipeline",
    "SARTAJ": "srijan2715/sartaj-final-split"
}
SEEDS = [42, 43, 44]

# ==========================================
# STEP 3: Bulletproof Data Transform
# ==========================================
def eval_transform(examples):
    images = examples['image']
    labels = examples['label']
    
    processed_images = []
    for img in images:
        if not isinstance(img, torch.Tensor):
            img = torch.tensor(img, dtype=torch.float32)
        else:
            img = img.float()
        img = img / 255.0
        
        if img.dim() == 3 and img.shape[-1] == 3:
            img = img.permute(2, 0, 1)
        processed_images.append(img)

    processed_labels = []
    for lbl in labels:
        if isinstance(lbl, str):
            # Use the exact mapping from the JSON file
            processed_labels.append(CLASS_TO_IDX[lbl])
        else:
            processed_labels.append(int(lbl))

    return {
        'image': torch.stack(processed_images),
        'label': torch.tensor(processed_labels, dtype=torch.long)
    }

# ==========================================
# STEP 4: Unified Evaluation Function
# ==========================================
def run_evaluation(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds

results = []
print("="*70)
print("🔬 STARTING COMPREHENSIVE EVALUATION (Train + Test Recovery)")
print("="*70)
print("⚠️  Note: Training accuracy is evaluated on raw images (no augmentations).")
print("    Values will be ~0.5-1% higher than live terminal logs. This is expected.\n")

for dataset_name, hf_id in DATASETS.items():
    for seed in SEEDS:
        filename = f"{dataset_name.lower()}_seed{seed}.pth"
        print(f"📦 Processing: {filename}...")
        
        try:
            # 1. Load Model
            model = ScratchCNN(num_classes=4)
            local_path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type="model")
            state_dict = torch.load(local_path, map_location=DEVICE)
            model.load_state_dict(state_dict)
            model.to(DEVICE)
            
            # 2. Evaluate TRAIN Set
            train_dataset = load_dataset(hf_id, split="train")
            train_dataset.set_transform(eval_transform)
            train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2)
            
            train_labels, train_preds = run_evaluation(model, train_loader)
            train_acc = accuracy_score(train_labels, train_preds)
            
            # 3. Evaluate TEST Set
            test_dataset = load_dataset(hf_id, split="test")
            test_dataset.set_transform(eval_transform)
            test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
            
            test_labels, test_preds = run_evaluation(model, test_loader)
            test_acc = accuracy_score(test_labels, test_preds)
            
            # Generate report using the correct display names
            report = classification_report(test_labels, test_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
            
            # 4. Store Results
            results.append({
                "Dataset": dataset_name,
                "Seed": seed,
                "Train Acc (%)": f"{train_acc*100:.2f}",
                "Test Acc (%)": f"{test_acc*100:.2f}",
                "Precision (Macro)": f"{report['macro avg']['precision']*100:.2f}",
                "Recall (Macro)": f"{report['macro avg']['recall']*100:.2f}",
                "F1-Score (Macro)": f"{report['macro avg']['f1-score']*100:.2f}"
            })
            
            print(f"  ✅ Success! Train: {train_acc*100:.2f}% | Test: {test_acc*100:.2f}%\n")
            
        except Exception as e:
            print(f"  ❌ FAILED: {e}\n")
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

# ==========================================
# STEP 5: Final Summary Table
# ==========================================
print("\n" + "="*70)
print("🏆 FINAL METRICS SUMMARY (Ready for Notebook) 🏆")
print("="*70)

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

print("\n💡 Tip: Highlight the table above, copy it, and paste it directly into your Markdown notebook!")

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json

print("--- SARTAJ NATIVE LABELS ---")
dataset = load_dataset("srijan2715/sartaj-final-split", split="train")
print("Unique labels in Sartaj:", set(dataset['label']))

print("\n--- SARTAJ LABEL MAPPING JSON ---")
for fname in ["label_mapping.json", "label_mapping1.json"]:
    try:
        path = hf_hub_download(repo_id="srijan2715/sartaj-final-split", filename=fname, repo_type="dataset")
        with open(path, 'r') as f:
            print(f"Found {fname}:", json.load(f))
    except:
        pass

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# 1. Define Model
class ScratchCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ScratchCNN, self).__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.gap(x); x = x.view(x.size(0), -1); x = self.classifier(x)
        return x

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]

# 2. Load Model
print("Loading sartaj_seed42.pth...")
model = ScratchCNN(num_classes=4)
path = hf_hub_download(repo_id="srijan2715/scratchcnn-checkpoints", filename="sartaj_seed42.pth", repo_type="model")
model.load_state_dict(torch.load(path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# 3. Load Sartaj Test Dataset
print("Loading Sartaj Test Set...\n")
dataset = load_dataset("srijan2715/sartaj-final-split", split="test")

# 4. Check the first 5 images
print("--- FIRST 5 PREDICTIONS ---")
for i in range(5):
    img_raw = dataset[i]['image']
    true_label_idx = dataset[i]['label'] # This is an integer: 0, 1, 2, or 3
    
    # Convert to tensor
    img_t = torch.tensor(img_raw, dtype=torch.float32)
    
    # Check if it needs scaling
    if img_t.max() > 1.0:
        img_t = img_t / 255.0
        
    # Permute to (C, H, W)
    if img_t.dim() == 3 and img_t.shape[-1] == 3:
        img_t = img_t.permute(2, 0, 1)
        
    # Add batch dimension and move to GPU
    img_t = img_t.unsqueeze(0).to(DEVICE)
    
    # Predict
    with torch.no_grad():
        logits = model(img_t)
        pred_idx = torch.argmax(logits, 1).item()
        
    print(f"Image {i+1}:")
    print(f"  True Label Index : {true_label_idx} ({CLASS_NAMES[true_label_idx]})")
    print(f"  Predicted Index  : {pred_idx} ({CLASS_NAMES[pred_idx]})")
    print(f"  Raw Logits       : {logits.cpu().numpy()[0].round(2)}")
    print(f"  Match?           : {'✅ YES' if true_label_idx == pred_idx else '❌ NO'}\n")

In [ ]:
from datasets import load_dataset
import hashlib
import torch

print("🔍 Scanning Sartaj Train and Test sets for leaked duplicates...")

# 1. Load Datasets
train_ds = load_dataset("srijan2715/sartaj-final-split", split="train")
test_ds = load_dataset("srijan2715/sartaj-final-split", split="test")

# 2. Hash all training images
print("Hashing training images...")
train_hashes = set()
for img in train_ds['image']:
    # Convert raw list to bytes for fast hashing
    t = torch.tensor(img, dtype=torch.uint8)
    train_hashes.add(hashlib.md5(t.numpy().tobytes()).hexdigest())

# 3. Find matches in the test set
print("Checking test set for leaks...")
leaked_indices = []
for i, img in enumerate(test_ds['image']):
    t = torch.tensor(img, dtype=torch.uint8)
    if hashlib.md5(t.numpy().tobytes()).hexdigest() in train_hashes:
        leaked_indices.append(i)

print(f"\n✅ FOUND {len(leaked_indices)} LEAKED IMAGES!")
print(f"Indices of leaked images in Test Set: {leaked_indices[:10]}... (showing first 10)")

# Save this list so the evaluation script can use it
LEAKED_INDICES = leaked_indices

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# 1. Define Model
class ScratchCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ScratchCNN, self).__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.gap(x); x = x.view(x.size(0), -1); x = self.classifier(x)
        return x

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]

# 2. Load Model
print("Loading sartaj_seed42.pth...")
model = ScratchCNN(num_classes=4)
path = hf_hub_download(repo_id="srijan2715/scratchcnn-checkpoints", filename="sartaj_seed42.pth", repo_type="model")
model.load_state_dict(torch.load(path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# 3. Load Sartaj Test Dataset
dataset = load_dataset("srijan2715/sartaj-final-split", split="test")

# 4. Find one image of each class (0, 1, 2, 3)
print("\n--- TESTING ONE IMAGE OF EACH CLASS ---")
found_classes = {0: None, 1: None, 2: None, 3: None}

for i in range(len(dataset)):
    lbl = dataset[i]['label']
    # Handle both string and integer labels just in case
    if isinstance(lbl, str):
        lbl_idx = {'glioma': 0, 'meningioma': 1, 'no': 2, 'no_tumor': 2, 'pituitary': 3}.get(lbl, int(lbl))
    else:
        lbl_idx = int(lbl)
        
    if found_classes[lbl_idx] is None:
        found_classes[lbl_idx] = i
        
    if all(v is not None for v in found_classes.values()):
        break

# 5. Run model on these 4 specific images
for class_idx, img_idx in found_classes.items():
    img_raw = dataset[img_idx]['image']
    
    # Transform
    img_t = torch.tensor(img_raw, dtype=torch.float32)
    if img_t.max() > 1.0: img_t = img_t / 255.0
    if img_t.dim() == 3 and img_t.shape[-1] == 3: img_t = img_t.permute(2, 0, 1)
    img_t = img_t.unsqueeze(0).to(DEVICE)
    
    # Predict
    with torch.no_grad():
        logits = model(img_t)
        pred_idx = torch.argmax(logits, 1).item()
        
    print(f"True Class: {class_idx} ({CLASS_NAMES[class_idx]})")
    print(f"  Predicted : {pred_idx} ({CLASS_NAMES[pred_idx]})")
    print(f"  Logits    : {logits.cpu().numpy()[0].round(2)}")
    print(f"  Match?    : {'✅ YES' if class_idx == pred_idx else '❌ NO'}\n")

In [ ]:
from huggingface_hub import HfApi, login

# Insert your token
login(token="secret") 

api = HfApi()
repo_id = "srijan2715/scratchcnn-checkpoints"

# Only delete the broken Sartaj files. The BRISC files are perfectly fine!
files_to_delete = [
    "sartaj_seed42.pth", 
    "sartaj_seed43.pth", 
    "sartaj_seed44.pth"
]

print("🧹 Cleaning up Hugging Face repository...")
for filename in files_to_delete:
    try:
        api.delete_file(path_in_repo=filename, repo_id=repo_id, repo_type="model")
        print(f"✅ Successfully deleted {filename}")
    except Exception as e:
        print(f"❌ Could not delete {filename}: {e}")

print("\n✨ Repository is now clean and ready for the final evaluation!")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import gc
import hashlib

# ==========================================
# 1. Architecture V2 (Exact match to E2E)
# ==========================================
class ScratchCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ScratchCNN, self).__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.gap(x); x = x.view(x.size(0), -1); x = self.classifier(x)
        return x

# ==========================================
# 2. Configuration
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {DEVICE}\n")

CLASS_TO_IDX = {'glioma': 0, 'meningioma': 1, 'pituitary': 3, 'no': 2, 'no_tumor': 2, 'notumor': 2, 'No Tumor': 2}
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]

REPO_ID = "srijan2715/scratchcnn-checkpoints"
DATASETS = {
    "SARTAJ": "srijan2715/sartaj-final-split",
    "BRISC": "srijan2715/brisc2025-paper-pipeline"
}
SEEDS = [42, 43, 44]

# ==========================================
# 3. Leakage Detection
# ==========================================
print("🔍 Scanning Sartaj for data leakage...")
sartaj_train = load_dataset("srijan2715/sartaj-final-split", split="train")
sartaj_test = load_dataset("srijan2715/sartaj-final-split", split="test")

train_hashes = set()
for img in sartaj_train['image']:
    # Robustly handle PIL or list
    arr = np.array(img) if hasattr(img, 'convert') else img
    train_hashes.add(hashlib.md5(torch.tensor(arr, dtype=torch.uint8).numpy().tobytes()).hexdigest())

LEAKED_INDICES = []
for i, img in enumerate(sartaj_test['image']):
    arr = np.array(img) if hasattr(img, 'convert') else img
    if hashlib.md5(torch.tensor(arr, dtype=torch.uint8).numpy().tobytes()).hexdigest() in train_hashes:
        LEAKED_INDICES.append(i)
print(f"✅ Found {len(LEAKED_INDICES)} leaked images in Sartaj test set.\n")

# ==========================================
# 4. 100% Bulletproof Transform
# ==========================================
def eval_transform(examples):
    images = examples['image']
    labels = examples['label']
    
    processed_imgs = []
    for img in images:
        # 1. Robustly convert PIL Image, list, or numpy array to tensor
        if hasattr(img, 'convert'):  # It's a PIL Image
            img_array = np.array(img)
        else:
            img_array = img
            
        t = torch.tensor(img_array, dtype=torch.float32)
        
        # 2. Normalize to [0, 1]
        if t.max() > 1.0:
            t = t / 255.0
            
        # 3. Ensure (C, H, W) format
        if t.dim() == 3 and t.shape[-1] == 3:
            t = t.permute(2, 0, 1)
            
        processed_imgs.append(t)

    processed_lbls = []
    for lbl in labels:
        # 4. SAFE label mapping (avoids evaluating int() on strings)
        if isinstance(lbl, str) and lbl in CLASS_TO_IDX:
            processed_lbls.append(CLASS_TO_IDX[lbl])
        else:
            processed_lbls.append(int(lbl))
            
    return {
        'image': torch.stack(processed_imgs), 
        'label': torch.tensor(processed_lbls, dtype=torch.long)
    }

def run_evaluation(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(batch['image'].to(DEVICE))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())
    return all_labels, all_preds

# ==========================================
# 5. Evaluation Loop
# ==========================================
results = []
print("="*70)
print("🏆 STARTING FINAL V2 EVALUATION (SARTAJ FIRST, THEN BRISC) 🏆")
print("="*70)

for dataset_name, hf_id in DATASETS.items():
    for seed in SEEDS:
        if dataset_name == "BRISC":
            filename = f"brisc_seed{seed}.pth"
        else:
            filename = f"sartaj_seed{seed}_v2.pth"
            
        print(f"\n📦 Processing: {filename}...")
        
        try:
            model = ScratchCNN(num_classes=4).to(DEVICE)
            local_path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type="model")
            model.load_state_dict(torch.load(local_path, map_location=DEVICE))
            
            # 1. Train Set
            train_ds = load_dataset(hf_id, split="train")
            train_ds.set_transform(eval_transform)
            train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=0)
            train_labels, train_preds = run_evaluation(model, train_loader)
            train_acc = accuracy_score(train_labels, train_preds)
            
            # 2. Official Test Set
            test_ds = load_dataset(hf_id, split="test")
            test_ds.set_transform(eval_transform)
            test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
            test_labels, test_preds = run_evaluation(model, test_loader)
            test_acc = accuracy_score(test_labels, test_preds)
            report = classification_report(test_labels, test_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
            
            results.append({
                "Dataset": dataset_name, "Seed": seed, "Test Set": "Official",
                "Train Acc (%)": f"{train_acc*100:.2f}", "Test Acc (%)": f"{test_acc*100:.2f}",
                "F1-Score": f"{report['macro avg']['f1-score']*100:.2f}"
            })
            
            # 3. Clean Test Set (Sartaj Only)
            if dataset_name == "SARTAJ" and len(LEAKED_INDICES) > 0:
                clean_indices = [i for i in range(len(test_ds)) if i not in LEAKED_INDICES]
                clean_ds = test_ds.select(clean_indices)
                clean_ds.set_transform(eval_transform)
                clean_loader = torch.utils.data.DataLoader(clean_ds, batch_size=64, shuffle=False, num_workers=0)
                clean_labels, clean_preds = run_evaluation(model, clean_loader)
                clean_acc = accuracy_score(clean_labels, clean_preds)
                clean_report = classification_report(clean_labels, clean_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
                
                results.append({
                    "Dataset": dataset_name, "Seed": seed, "Test Set": f"Clean ({len(clean_ds)})",
                    "Train Acc (%)": f"{train_acc*100:.2f}", "Test Acc (%)": f"{clean_acc*100:.2f}",
                    "F1-Score": f"{clean_report['macro avg']['f1-score']*100:.2f}"
                })
                print(f"  ✅ Official: {test_acc*100:.2f}% | Clean: {clean_acc*100:.2f}%")
            else:
                print(f"  ✅ Train: {train_acc*100:.2f}% | Test: {test_acc*100:.2f}%")
                
        except Exception as e:
            print(f"  ❌ FAILED: {e}")
        finally:
            del model; gc.collect(); torch.cuda.empty_cache()

# ==========================================
# 6. Final Table
# ==========================================
print("\n" + "="*70)
print("🎉 FINAL METRICS SUMMARY 🎉")
print("="*70)
print(pd.DataFrame(results).to_markdown(index=False))